**Preprocessing & EDA: Raw CSIC-2010 CSV to Request-Level Dataset**  

Explore the raw CSIC 2010 CSV and reconstruct one row per HTTP request.

**Method:** 
- Load the raw CSV and inspect columns, types, and nulls
- Drop constant columns and identifier columns
- Deduce the request key from repeated index values
- Reconstruct requests by grouping rows on cookie and joining payload fragments
- Verify the reconstruction: request count and class split match the source dataset

**Inputs:**
- `data/csic_2010_raw.csv` - `223,585` rows × `18` columns; downloaded from the hosted copy

**Outputs:**
- `data/csic_2010_preprocessed.parquet` - `61,065` requests × `8` columns (`method`, `url`, `host`, `payload`, `contentLength`, `label`, `cookie`, `index`)

In [21]:
import os
from pathlib import Path
import pandas as pd
import urllib.request

In [22]:
# working directory for relative paths
notebooks_path = Path.cwd() # <repo-root>/notebooks
repo_root_path = notebooks_path.parent # <repo-root>/
data_path = repo_root_path / "data" # <repo-root>/data
input_dataset_path = data_path / "csic_2010_raw.csv" # input: raw dataset 
output_dataset_path = data_path / "csic_2010_preprocessed.parquet" # output: processed dataset

**Download the raw dataset**  
The original dataset is the [CSIC 2010 web attack dataset](http://lexr.ai/csic_dataset/output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv.zip). For ease of access and reproducibility, a copy is hosted on Hugging Face and is downloaded below

In [23]:
raw_url = "https://huggingface.co/datasets/arbk2023/csic_2010/resolve/main/output_http_csic_2010_weka_with_duplications_RAW-RFC2616_escd_v02_full.csv"

if input_dataset_path.exists():
    print(f"Dataset already present: {input_dataset_path} — skipping download.")
else:
    os.makedirs(os.path.dirname(input_dataset_path), exist_ok=True)
    urllib.request.urlretrieve(raw_url, input_dataset_path)
    print(f"Downloaded dataset to: {input_dataset_path}")   

Dataset already present: /mnt/hgfs/workspace/mlhttp/data/csic_2010_raw.csv — skipping download.


**Load the dataset**

Load `csic_2010_raw.csv` and inspect columns, shape and few sample of rows

In [24]:
df_raw = pd.read_csv(input_dataset_path, low_memory=False)
print(f"shape: {df_raw.shape}")

shape: (223585, 18)


In [25]:
# columns and sample rows
df_raw

,index,method,url,protocol,userAgent,pragma,cacheControl,accept,acceptEncoding,acceptCharset,acceptLanguage,host,connection,contentLength,contentType,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,NaN,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223580,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ciudad=Hontangas,norm
223581,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,cp=08192,norm
223582,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,provincia=Teruel,norm
223583,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,HTTP/1.1,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,close,238.0,application/x-www-form-urlencoded,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ntc=8231060336250168,norm


**Inspect columns and data types**

In [26]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223585 entries, 0 to 223584
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   index           223585 non-null  int64  
 1   method          223585 non-null  object 
 2   url             223585 non-null  object 
 3   protocol        223585 non-null  object 
 4   userAgent       223585 non-null  object 
 5   pragma          223585 non-null  object 
 6   cacheControl    223585 non-null  object 
 7   accept          223585 non-null  object 
 8   acceptEncoding  223585 non-null  object 
 9   acceptCharset   223585 non-null  object 
 10  acceptLanguage  223585 non-null  object 
 11  host            223585 non-null  object 
 12  connection      223585 non-null  object 
 13  contentLength   100135 non-null  float64
 14  contentType     100135 non-null  object 
 15  cookie          223585 non-null  object 
 16  payload         198094 non-null  object 
 17  label     

**Observations:**  
- Dataset contains `223,585` rows and 18 columns.  
- `contentLength`, `contentType`, `payload` columns have values missing.  

**Inspect value counts per column**

In [27]:
n_unique = df_raw.nunique()
n_unique

index             36000
method                3
url                1643
protocol              1
userAgent             1
pragma                1
cacheControl          1
accept                1
acceptEncoding        1
acceptCharset         1
acceptLanguage        1
host                  2
connection            1
contentLength       382
contentType           1
cookie            61065
payload           33769
label                 2
dtype: int64

**Observations:**  
- `index` column shows `36000` unique values whereas total rows count is `223585`, It implies that index column values are not unique. 
- `method` column with `3` unique values (GET, POST, PUT), Can be used as Categorial feature (one-hot encode)
- `url` with `1643` unique values. Can be used to derive various features columns. eg: url_length, count of special characters, presense of malicious patterns etc
- `host` column has `2` unique values. Can be used as Categorical feature (one-hot encode)
- `contentLength` column has `382` unique values. Can be used as numeric feature column.
- `cookie` column has `61065` unique values. It is string identifier.
- `payload` column with `33769` unique values. Can be used to derive various features columns. eg: payload_length, count of special characters, presense of malicious patterns etc
- `protocol`, `userAgent`, `pragma`, `cacheControl`, `accept`, `acceptEncoding`, `acceptCharset`, `acceptLanguage`, `connection`, `contentType` all these columns have a single unique value. These columns can be dropped.
- `label` column has `2` values. Can be used as Categorical feature
- `index` and `cookie` look like identifiers.

**Drop constant columns**

In [28]:
const_cols_list = n_unique[n_unique == 1].index
const_cols = list(const_cols_list)
print(const_cols)

['protocol', 'userAgent', 'pragma', 'cacheControl', 'accept', 'acceptEncoding', 'acceptCharset', 'acceptLanguage', 'connection', 'contentType']


In [29]:
df_multi = df_raw.drop(columns = const_cols)
df_multi

,index,method,url,host,contentLength,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom
...,...,...,...,...,...,...,...,...
223580,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ciudad=Hontangas,norm
223581,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,cp=08192,norm
223582,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,provincia=Teruel,norm
223583,35999,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,238.0,JSESSIONID=05C0977CEA3691A57DFC74F2A03C3D2C,ntc=8231060336250168,norm


Initial row-level analysis suggests `index` values repeat. Inspect all rows for a single index value to understand what the repetition means.

**Analyze repeated index values**

Analyze rows for `index=0`

In [30]:
group_index0 = df_multi[df_multi['index'] == 0]
group_index0

,index,method,url,host,contentLength,cookie,payload,label
0,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,id=2,anom
1,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,nombre=Jam%F3n+Ib%E9rico,anom
2,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,precio=85,anom
3,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+...,anom
4,0,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=B92A8B48B9008CD29F622A994E0F650D,B1=A%F1adir+al+carrito,anom
119585,0,GET,http://localhost:8080/tienda1/index.jsp,localhost:8080,NaN,JSESSIONID=1F767F17239C9B670A39E9B10C3825F4,NaN,norm


**Observations:**  

`index=0` appears in two non-contiguous groups of rows:

- **Group 1** (rows 0-4, `anom`): all rows share `cookie`, `label`, `method`, `url`, `host`, `contentLength`, but each row holds a different fragment of the payload.  
- **Group 2** (row 119585, `norm`): The row differs in `url` and `cookie` from **Group 1**. It is a different request that happens to carry the same `index` value. The payload is `NaN` (body-less request).

Analyze rows for `index=1`

In [31]:
group_index1 = df_multi[df_multi['index'] == 1]
group_index1

,index,method,url,host,contentLength,cookie,payload,label
5,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,id=2%2F,anom
6,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,nombre=Jam%F3n+Ib%E9rico,anom
7,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,precio=85,anom
8,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,cantidad=49,anom
9,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=F563B5262843F12ECAE41815ABDEEA54,B1=A%F1adir+al+carrito,anom
119586,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,id=3,norm
119587,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,nombre=Vino+Rioja,norm
119588,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,precio=100,norm
119589,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,cantidad=55,norm
119590,1,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,NaN,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,B1=A%F1adir+al+carrito,norm


**Observations:**  

`index=1` appears in two non-contiguous groups of rows:

- **Group 1** (rows 5-9, `anom`): all rows share `cookie`, `label`, `method`, `url`, `host`, `contentLength`, but each row holds a different fragment of the payload.  
- **Group 2** (rows 119586–119590, `norm`): all rows share `cookie`, `label`, `method`, `url`, `host`, `contentLength`, but each row holds a different fragment of the payload.

Based on the analysis of `index=0` and `index=1` it can be deduced that rows are fragments of HTTP requests, and column combination identifies which request a row belongs to.

**Identify the HTTP Request Key**

Count uniques values of (`cookie` & `index`) pair and `cookie` only 

In [32]:
total_requests = df_multi.groupby(["cookie", "index"]).ngroups

print(f"unique cookies:           {df_multi['cookie'].nunique()}")
print(f"unique (cookie, index):   {total_requests}")
print(f"total rows:               {len(df_multi)}")

unique cookies:           61065
unique (cookie, index):   61065
total rows:               223585


**Observations:**  
- `61,065` unique `cookies`, and `61,065` unique `(cookie, index)` pairs. So cookie alone identifies a request. `index` adds nothing.
- The `223,585` rows are fragments across `61,065` requests.
- `index` is a per-request value, reused across different requests.  
- Single HTTP request spans multiple consecutive rows with fragmented payload
- All rows of a request share `label`, `method`, `url`, `host`, `contentLength`.

**Conclusion:**  
- Rows must be grouped by `cookie` and payloads reassembled to make up a single HTTP request
- Reconstruction must preserve the row count of `61,065` requests and the class split of `36,000` norm / `25,065` anom

**Reconstruct HTTP requests**

Group by `cookie`, take the shared fields from the first row, and concatenate payload fragments

In [33]:
# Group the Data Frame by 'cookie'
grouped_by_cookie = df_multi.groupby('cookie')
df_reconstructed = (
    grouped_by_cookie.agg({
        'index': 'first',  # Take the first index (all rows in the group share the same index)
        'method': 'first',  # Take the first method (all rows in the group share the same method)
        'url': 'first',     # Take the first URL (all rows in the group share the same url)
        'host': 'first',     # Take the first host (all rows in the group share the same host)
        'label': 'first',   # Take the first label (all rows in the group share the same label)
        'payload': lambda x: ' '.join(x.dropna()),  # Combine all non-NaN payload fragments
        'contentLength': 'first', # Take the first contentLength (all rows in the group share the same contentLength)
    })
    .reset_index()  # Convert 'cookie' from index to a column
)

**Validate the reconstruction**

In [34]:
print(f"shape: {df_reconstructed.shape}")
print(df_reconstructed["label"].value_counts())

shape: (61065, 8)
label
norm    36000
anom    25065
Name: count, dtype: int64


In [35]:
df_reconstructed

,cookie,index,method,url,host,label,payload,contentLength
0,JSESSIONID=0000442E26C2F96F52C97120FF00254B,6883,GET,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,NaN
1,JSESSIONID=0000AF0F87A92C3DFEE827BDFC243736,4687,GET,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,NaN
2,JSESSIONID=00020FC2706BF9F82BE6D4A7BBE12F41,25623,GET,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,NaN
3,JSESSIONID=0002AB6A7AC61AE9DC71925807460D8C,10367,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,NaN
4,JSESSIONID=0003D3946ADEBD8EE20F3B4700193661,19522,POST,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0
...,...,...,...,...,...,...,...,...
61060,JSESSIONID=FFFE094F84AF5FA4FF65367134F7FA9A,35211,POST,http://localhost:8080/tienda1/publico/entrar.jsp,localhost:8080,norm,errorMsg=Credenciales+incorrectas,33.0
61061,JSESSIONID=FFFE2D706D5003EFE6113C6B721FDD97,18514,POST,http://localhost:8080/tienda1/miembros/editar.jsp,localhost:8080,anom,modo=registro login=fabris password=dEsEnquieT...,248.0
61062,JSESSIONID=FFFEC3CFA8BEA96ADBAB57FABFA311C2,5875,GET,http://localhost:8080/tienda1/publico/autentic...,localhost:8080,anom,modo=entrar login=quoi pwdA=gAMar9a remember=o...,NaN
61063,JSESSIONID=FFFF432CCE806C813B7A1CA89B874517,31132,POST,http://localhost:8080/tienda1/publico/pagar.jsp,localhost:8080,norm,modo=insertar precio=4760 B1=Confirmar,38.0


`df_reconstructed` data frame contains `61,065` rows (**reconstructed HTTP requests**) with `8` columns.

Inspect the structure of reconstructed dataframe

In [36]:
df_reconstructed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cookie         61065 non-null  object 
 1   index          61065 non-null  int64  
 2   method         61065 non-null  object 
 3   url            61065 non-null  object 
 4   host           61065 non-null  object 
 5   label          61065 non-null  object 
 6   payload        61065 non-null  object 
 7   contentLength  17977 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.7+ MB


**Observations:**  
- The core metadata columns `cookie`, `index`, `method`, `url`, `host`, `label` and `payload` are fully populated (`61,065` non-null entries). This confirms that each reconstructed HTTP request retains complete metadata. The grouping by `cookie` preserves the integrity of the original requests.  
- `contentLength` is populated for only `17,977` of `61,065` requests 

Content-Length header is only sent when a request carries a body. Only POST and PUT carry bodies. GET requests do not.

Check that null `contentLength` aligns with GET requests and non-null with POST/PUT

In [37]:
pd.crosstab(df_reconstructed["method"], df_reconstructed["contentLength"].notna(), margins=True)

contentLength,False,True,All
method,,,
GET,43088,0,43088
POST,0,17580,17580
PUT,0,397,397
All,43088,17977,61065


**Observations:**  
- All `43,088` GET requests lack `contentLength` (no request body)
- All `17,580` POST and `397` PUT requests have non-null `contentLength`

One row is now one HTTP request. All fields are populated except `contentLength`, which is correctly absent for body-less GET requests. This is the right structure for feature engineering.

**Store the reconstructed dataset to disk**

Persist the request-level dataset (61,065 rows × 8 columns) as `csic_2010_preprocessed.parquet` for feature engineering in `featurise.ipynb`.

In [38]:
# Save as Parquet
%time df_reconstructed.to_parquet(output_dataset_path)
print(f"stored: {output_dataset_path}")

CPU times: user 40.4 ms, sys: 208 ms, total: 249 ms
Wall time: 283 ms
stored: /mnt/hgfs/workspace/mlhttp/data/csic_2010_preprocessed.parquet


In [39]:
body = df_reconstructed[df_reconstructed["method"] != "GET"]
gap = (body["payload"].str.len() - body["contentLength"])
print(gap.value_counts().head())

0.0    17977
Name: count, dtype: int64
